In [1]:
import pandas as pd

dataset = pd.read_csv("augmented_dataset.csv")

/tmp/ipykernel_777670/20200156.py:3: DtypeWarning: Columns (5,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset = pd.read_csv("augmented_dataset.csv")


In [2]:
dataset.drop_duplicates(subset="SMILES").to_csv("unique_compounds.csv", index=False)

In [ ]:
import os
from deepmol.loaders import CSVLoader
from deepmol.compound_featurization import ThreeDimensionalMoleculeGenerator

# Processing parameters
timeout = 200
threads = 50
n_conformations = 1
max_iterations = 100
etkdg_version = 3
mode = "MMFF94"

dataset = CSVLoader("unique_compounds.csv", id_field="Substrate ID", smiles_field="SMILES").create_dataset()
            
# Generate 3D conformers
generator = ThreeDimensionalMoleculeGenerator(
    timeout_per_molecule=timeout, threads=threads,
    n_conformations=n_conformations, max_iterations=max_iterations
)
generator.generate(dataset, etkdg_version=etkdg_version, mode=mode)

# Save as SDF
output_sdf_path = os.path.join("unique_compounds_conformers.sdf")
dataset.to_sdf(output_sdf_path)

In [11]:
from deepmol.loaders._utils import load_csv_file, load_sdf_file
import pandas as pd

mols = load_sdf_file("unique_compounds_conformers.sdf")
ids = []
for mol in mols:
    id_ = mol.GetProp("_ID")
    ids.append(id_)

df = pd.DataFrame({"ids":ids, "mols": mols})

[22:08:33] WARNING: not removing hydrogen atom without neighbors
[22:08:33] Warning: ambiguous stereochemistry - zero final chiral volume - at atom 8 ignored
[22:08:33] Warning: ambiguous stereochemistry - linear bond arrangement - at atom 11 ignored
[22:08:33] Warning: ambiguous stereochemistry - linear bond arrangement - at atom 5 ignored
[22:08:33] WARNING: not removing hydrogen atom with dummy atom neighbors


In [12]:
df = df.drop_duplicates(subset="ids")

In [13]:
from deepmol.datasets import SmilesDataset

dataset = SmilesDataset(smiles=df.mols, mols=df.mols, ids=df.ids)
dataset.to_sdf("unique_compounds_conformers.sdf")

In [14]:
from deepmol.loaders import SDFLoader

compounds_dataset = SDFLoader("unique_compounds_conformers.sdf", id_field="_ID").create_dataset()

[22:08:48] WARNING: not removing hydrogen atom without neighbors
[22:08:48] WARNING: not removing hydrogen atom with dummy atom neighbors
[22:08:48] Warning: ambiguous stereochemistry - linear bond arrangement - at atom 61 ignored
[22:08:48] Warning: ambiguous stereochemistry - linear bond arrangement - at atom 69 ignored
[22:08:48] Warning: ambiguous stereochemistry - linear bond arrangement - at atom 61 ignored
[22:08:48] Warning: ambiguous stereochemistry - linear bond arrangement - at atom 71 ignored
[22:08:48] Warning: ambiguous stereochemistry - linear bond arrangement - at atom 82 ignored
[22:08:48] Warning: ambiguous stereochemistry - linear bond arrangement - at atom 95 ignored
[22:08:48] Warning: ambiguous stereochemistry - linear bond arrangement - at atom 61 ignored
[22:08:48] Warning: ambiguous stereochemistry - linear bond arrangement - at atom 69 ignored
[22:08:48] Warning: ambiguous stereochemistry - linear bond arrangement - at atom 61 ignored
[22:08:48] Warning: ambig

3905
3905


In [15]:
compounds_dataset.ids

array(['compound_aminotransferase_dataset 1',
       'compound_aminotransferase_dataset 2',
       'compound_aminotransferase_dataset 3', ..., 'C06026',
       'CHEBI:58677', 'CHEBI:10329'], dtype='<U43')

In [16]:
compounds_dataset.get_shape()

2025-04-11 22:09:04,124 — INFO — Mols_shape: (3905,)
2025-04-11 22:09:04,125 — INFO — Features_shape: None
2025-04-11 22:09:04,126 — INFO — Labels_shape: None


((3905,), None, None)

In [17]:
from deepmol.compound_featurization import All3DDescriptors, MixedFeaturizer, TwoDimensionDescriptors

MixedFeaturizer([All3DDescriptors(mandatory_generation_of_conformers=False), TwoDimensionDescriptors()]).featurize(compounds_dataset, inplace=True)

2025-04-11 22:11:13,460 — ERROR — Failed to featurize *C(=O)NC(CO[C@@H]1O[C@H](CO)[C@@H](O[C@@H]2O[C@H](CO)[C@H](O[C@@H]3O[C@H](CO)[C@H](O)[C@H](O[C@@H]4O[C@H](CO)[C@H](O)[C@H](O[C@]5(C(=O)[O-])C[C@H](O)[C@@H](NC(C)=O)[C@H]([C@H](O)[C@H](O)CO)O5)[C@H]4O)[C@H]3NC(C)=O)[C@H](O[C@]3(C(=O)[O-])C[C@H](O)[C@@H](NC(C)=O)[C@H]([C@H](O)[C@H](O)CO)O3)[C@H]2O)[C@H](O)[C@H]1O)[C@H](O)/C=C/CCCCCCCCCCCCC. Appending empty array
2025-04-11 22:11:13,486 — ERROR — Exception message: 
2025-04-11 22:11:13,994 — ERROR — Failed to featurize *C(=O)NC(CO[C@@H]1O[C@H](CO)[C@@H](O[C@@H]2O[C@H](CO)[C@H](O[C@@H]3O[C@H](CO)[C@H](O)[C@H](O[C@@H]4O[C@H](CO)[C@H](O)[C@H](O[C@]5(C(=O)[O-])C[C@H](O)[C@@H](NC(C)=O)[C@H]([C@H](O)[C@H](O)CO)O5)[C@H]4O)[C@H]3NC(C)=O)[C@H](O[C@]3(C(=O)[O-])C[C@H](O)[C@@H](NC(C)=O)[C@H]([C@H](O)C(CO)O[C@]4(C(=O)[O-])C[C@H](O)[C@@H](NC(C)=O)[C@H]([C@H](O)[C@H](O)CO)O4)O3)[C@H]2O)[C@H](O)[C@H]1O)[C@H](O)/C=C/CCCCCCCCCCCCC. Appending empty array
2025-04-11 22:11:13,996 — ERROR — Exception messa

MixedFeaturizer: 100%|██████████| 3905/3905 [02:04<00:00, 31.30it/s]


In [19]:
compounds_dataset.get_shape()

2025-04-11 22:11:19,109 — INFO — Mols_shape: (3298,)
2025-04-11 22:11:19,110 — INFO — Features_shape: (3298, 849)
2025-04-11 22:11:19,110 — INFO — Labels_shape: None


((3298,), (3298, 849), None)

In [20]:
compounds_dataset.to_sdf(path="unique_compounds_with_features.sdf")

In [21]:
import pandas as pd

dataset = pd.read_csv("augmented_dataset.csv")
dataset

/tmp/ipykernel_1262493/4037893231.py:3: DtypeWarning: Columns (5,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset = pd.read_csv("augmented_dataset.csv")


,Sequence,SMILES,Binding,Enzyme ID,Substrate ID,Publication,pubchem_id,Validated,RHEA_ID,EC number,reaction_SMILES
0,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,C(C(=O)O)N,0.0,P00509,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0,True,NaN,NaN,NaN
1,MDYVTLASHAVRQYAPDQIFTASQRAKADAAALGEDAVINATLGEC...,C(C(=O)O)N,0.0,A0A174XK40,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0,True,NaN,NaN,NaN
2,MFQKVDAYAGDPILTLMERFKEDPRSDKVNLSIGLYYNEDGIIPQL...,C(C(=O)O)N,0.0,P04693,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0,True,NaN,NaN,NaN
3,MFQKVDAYAGDPILSLMERFKEDPRSDKVNLSIGLYYNDDGIIPQL...,C(C(=O)O)N,0.0,A0A3N0RMA5,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0,True,NaN,NaN,NaN
4,MSPIEKSSKLENVCYDIRGPVLKEAKRLEEEGNKVLKLNIGNPAPF...,C(C(=O)O)N,0.0,P0A959,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0,True,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1015426,MEPEAPRRRHTHQRGYLLTRNPHLNKDLAFTLEERQQLNIHGLLPP...,Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)([O-])OP(=O...,0.0,P48163,CHEBI:30616,NaN,NaN,False,NaN,NaN,NaN
1015427,MEPEAPRRRHTHQRGYLLTRNPHLNKDLAFTLEERQQLNIHGLLPP...,Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)(O)OP(=O)(O...,0.0,P48163,C00002,NaN,NaN,False,NaN,NaN,NaN
1015428,DGLRGGGRFFHGLAWILEEPWSSWFILNLKTTAEEEKEEVESLKSI...,COC1=C(OC)C(=O)C(CC=C(C)C)=C(C)C1=O,0.0,Q3TIG7,CHEBI:16389,NaN,NaN,False,NaN,NaN,NaN
1015429,DGLRGGGRFFHGLAWILEEPWSSWFILNLKTTAEEEKEEVESLKSI...,N[C@@H](CCC([O-])=N[C@@H](CS)C(O)=NCC(=O)O)C(=O)O,0.0,Q3TIG7,CHEBI:57925,NaN,NaN,False,NaN,NaN,NaN


In [22]:
dataset[dataset["Substrate ID"].isin(compounds_dataset.ids)].to_csv("augmented_dataset_descriptors_available.csv", index=False)

In [23]:
compounds_dataset.to_csv("unique_compounds_with_features.csv", index=False)

In [24]:
import pandas as pd
compounds_with_features = pd.read_csv("unique_compounds_with_features.csv")
compounds_with_features

,ids,smiles,Asphericity,AUTOCORR3D_0,AUTOCORR3D_1,AUTOCORR3D_2,AUTOCORR3D_3,AUTOCORR3D_4,AUTOCORR3D_5,AUTOCORR3D_6,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,compound_aminotransferase_dataset 1,NCC(=O)O,0.408650,0.557,0.955,0.648,0.000,0.000,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,compound_aminotransferase_dataset 2,C[C@H](N)C(=O)O,0.273338,0.474,0.970,0.863,0.000,0.000,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,compound_aminotransferase_dataset 3,CC(C)[C@H](N)C(=O)O,0.100952,0.364,0.795,0.935,0.521,0.000,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,compound_aminotransferase_dataset 4,CC(C)C[C@H](N)C(=O)O,0.442271,0.326,0.687,0.748,0.748,0.600,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,compound_aminotransferase_dataset 5,CC[C@H](C)[C@H](N)C(=O)O,0.278073,0.326,0.692,0.921,0.718,0.278,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3293,CHEBI:58158,[NH3+]C(C(=O)[O-])C(=O)[O-],0.215313,0.347,0.770,0.898,0.549,0.000,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3294,CHEBI:78435,CC(C)=CCC/C(C)=C/CC/C(C)=C/CC/C(C)=C\CC/C(C)=C...,0.920867,0.023,0.054,0.079,0.107,0.138,0.175,0.195,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3295,C15778,C=C(CC[C@@H](C)[C@H]1CC[C@H]2C3=CC=C4C[C@@H](O...,0.745251,0.120,0.312,0.487,0.552,0.558,0.595,0.618,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3296,CHEBI:58677,C[C@H](CCC[C@@H](C)[C@H]1CC[C@H]2[C@@H]3[C@H](...,0.754632,0.041,0.106,0.157,0.203,0.233,0.247,0.281,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [25]:
from plants_sm.io.pickle import write_pickle

features_dict = compounds_with_features.set_index('ids').drop(columns='smiles').apply(lambda row: row.to_numpy(), axis=1).to_dict()
write_pickle("compounds_features.pkl", features_dict)

True

In [26]:
import pandas as pd

dataset = pd.read_csv("augmented_dataset_descriptors_available.csv")
dataset["Enzyme ID"] = dataset["Enzyme ID"].str.replace(" ", "_")
dataset.to_csv("augmented_dataset_descriptors_available.csv", index=False)

/tmp/ipykernel_1262493/982059309.py:3: DtypeWarning: Columns (5,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset = pd.read_csv("augmented_dataset_descriptors_available.csv")


In [27]:
dataset.Binding.value_counts()

0.0    518892
1.0    301029
Name: Binding, dtype: int64

In [28]:
dataset.sample(frac=0.1).to_csv("test_integrated_dataset.csv")

In [29]:
dataset.shape

(819921, 11)

In [30]:
dataset.drop_duplicates(subset=["Sequence"]).loc[:, ["Sequence", "Enzyme ID"]].to_csv("unique_enzymes.csv", index=False)

In [32]:
dataset.drop_duplicates(subset=["Sequence"]).loc[:, ["Sequence", "Enzyme ID"]]

,Sequence,Enzyme ID
0,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,P00509
1,MDYVTLASHAVRQYAPDQIFTASQRAKADAAALGEDAVINATLGEC...,A0A174XK40
2,MFQKVDAYAGDPILTLMERFKEDPRSDKVNLSIGLYYNEDGIIPQL...,P04693
3,MFQKVDAYAGDPILSLMERFKEDPRSDKVNLSIGLYYNDDGIIPQL...,A0A3N0RMA5
4,MSPIEKSSKLENVCYDIRGPVLKEAKRLEEEGNKVLKLNIGNPAPF...,P0A959
...,...,...
819797,MADSHSVNNQLSRHTSIFGLRLWVVLGVCVGAAIVLLLVLISLWFI...,O22764
819821,MNLKQFTCLSCAQLLAILLFIFAFFPRKIVLTGISKQDPDQDRDLQ...,P40367
819840,MKTETHNRQCIPLAISVLSLFINGVSSARQPPDRCNRVCGEISIPF...,Q7X8C5
819875,MGSSEPLPIVDSDKRRKKKRKTRATDSLPGKFEDVYQLTSELLGEG...,Q4G050


In [31]:
import csv

def csv_to_fasta(csv_file_path, fasta_file_path):
    with open(csv_file_path, mode='r', newline='') as csv_file, open(fasta_file_path, mode='w') as fasta_file:
        csv_reader = csv.reader(csv_file)
        next(csv_reader)  # Skip the header row if there is one

        for row in csv_reader:
            if len(row) >= 2:
                sequence_id = row[1].replace(" ", "_")
                sequence = row[0]
                fasta_file.write(f">{sequence_id}\n")
                fasta_file.write(f"{sequence}\n")

# Example usage
csv_file_path = 'unique_enzymes.csv'  # Path to your CSV file
fasta_file_path = 'unique_enzymes.fasta'  # Path to save the FASTA file
csv_to_fasta(csv_file_path, fasta_file_path)

In [5]:
import pandas as pd

dataset = pd.read_csv("integrated_dataset_descriptors_available.csv")
from plants_sm.io.pickle import read_pickle
splits = read_pickle("splits/splits_0_6_proteins.pkl")


In [24]:
import pickle
# Save the pickle file with a compatible protocol version
with open('splits_0_6_proteins_4.pkl', 'wb') as f:
    pickle.dump(splits, f, protocol=4)  # Use protocol 4 for compatibility with Python 3.4+

In [19]:
train_dataset = dataset[dataset["Enzyme ID"].isin(splits[0][0])]
train_dataset

,Sequence,SMILES,Binding,Enzyme ID,Substrate ID,Publication,pubchem_id
2,MFQKVDAYAGDPILTLMERFKEDPRSDKVNLSIGLYYNEDGIIPQL...,C(C(=O)O)N,0.0,P04693,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0
3,MFQKVDAYAGDPILSLMERFKEDPRSDKVNLSIGLYYNDDGIIPQL...,C(C(=O)O)N,0.0,A0A3N0RMA5,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0
4,MSPIEKSSKLENVCYDIRGPVLKEAKRLEEEGNKVLKLNIGNPAPF...,C(C(=O)O)N,0.0,P0A959,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0
5,MADTRPERRFTRIDRLPPYVFNITAELKMAARRRGEDIIDFSMGNP...,C(C(=O)O)N,0.0,P77434,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0
9,MVITIDMLNPNVIETQYAVRGPIIILAQELEKKGKKMYYFNIGNPQ...,C(C(=O)O)N,0.0,A0A1Q9NYW4,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0
...,...,...,...,...,...,...,...
74301,MKYSRVFINSLAYELALVVVSSSELESRLAPLYQKFRIPMGQLAAL...,C1=CC=C(C=C1)C(=O)OC2=CC=C(C=C2)[N+](=O)[O-],0.0,Shewanella_putrefaciens,benzoate,https://doi.org/10.1093/synbio/ysaa004,70396.0
74302,MVKTMNYIAKLVGVGQALPERVVTSEEMEAMMGFDQLGIRKGMAKI...,C1=CC=C(C=C1)C(=O)OC2=CC=C(C=C2)[N+](=O)[O-],0.0,Anaerocolumna_xylanovorans,benzoate,https://doi.org/10.1093/synbio/ysaa004,70396.0
74304,MAFLSVNNVEIVGLAAAVPKNVETLDNLEFFAPGEAEKVMALTGIK...,C1=CC=C(C=C1)C(=O)OC2=CC=C(C=C2)[N+](=O)[O-],0.0,Bacteroides_luti,benzoate,https://doi.org/10.1093/synbio/ysaa004,70396.0
74305,MSAPRYSQVSAVAVRLPDEDLTTPELEELLAERNPRVDVPRGLIER...,C1=CC=C(C=C1)C(=O)OC2=CC=C(C=C2)[N+](=O)[O-],0.0,Nonomuraea_candida,benzoate,https://doi.org/10.1093/synbio/ysaa004,70396.0


In [1]:
import pandas as pd

pd.read_csv("descriptors_np_classifier.csv").iloc[:, :12]

,Model type,Trial,F1_macro_validation_set,Precision validation set,recall validation set,roc_auc validation set,accuracy validation set,mcc validation set,F1_macro_test_set,Precision test set,recall test set,roc_auc test set
0,gat,0,0.784405,0.878699,0.709895,0.902939,0.866757,0.698750,0.779013,0.871127,0.706732,0.901145
1,gat,1,0.779769,0.877868,0.702229,0.901146,0.864305,0.692890,0.776099,0.874065,0.699308,0.898639
2,gat,2,0.789861,0.858401,0.732208,0.906273,0.866444,0.697827,0.786602,0.862153,0.727643,0.902518
3,gat,3,0.310022,0.378258,0.263387,0.661423,0.741414,0.284569,0.304009,0.373163,0.260237,0.653892
4,gat,4,0.000000,0.000000,0.000000,0.500000,0.658024,0.000000,0.000000,0.000000,0.000000,0.500000
5,gat,5,0.787161,0.875373,0.715885,0.907168,0.867446,0.700187,0.778844,0.866739,0.708606,0.899359
6,gat,6,0.792892,0.891335,0.714345,0.908996,0.872252,0.711278,0.788960,0.885579,0.712664,0.906114
7,gat,7,0.775426,0.882697,0.694143,0.895986,0.862589,0.690373,0.776346,0.888434,0.690417,0.896530
8,gat,8,0.793009,0.888714,0.720543,0.906178,0.871689,0.712307,0.789309,0.880925,0.716507,0.907049
9,gat,9,0.780861,0.884262,0.701502,0.898238,0.865573,0.696900,0.773288,0.858315,0.704222,0.896388


In [2]:
import pandas as pd

pd.read_csv("binding_np_classifier.csv").sort_values(by="mcc validation set", ascending=False)

,Model type,Trial,F1_macro_validation_set,Precision validation set,recall validation set,roc_auc validation set,accuracy validation set,mcc validation set,F1_macro_test_set,Precision test set,...,accuracy test set,mcc test set,Batch size,Learning rate,Protein layers,Compound layers,Final head layers,Epochs,batch_norm,dropout
9,gat,9,0.773401,0.965712,0.646385,0.899672,0.870568,0.715895,0.771773,0.963233,...,0.869591,0.713224,16,0.000221,"[2560, 2560, 2560, 1280, 640]","[2560, 1280]","[2560, 1280, 1280, 1280]",128,False,0.779245
10,gat,10,0.789174,0.889518,0.713324,0.905458,0.870089,0.708294,0.784059,0.874473,...,0.865422,0.696391,32,0.000080,"[2560, 2560, 2560, 1280, 640]","[2560, 1280, 1280, 640]","[2560, 1280, 1280, 1280]",80,True,0.588846
0,gat,0,0.788584,0.892743,0.707242,0.897903,0.870057,0.706777,0.776560,0.876913,...,0.862593,0.689376,64,0.000547,"[2560, 1280]","[2560, 1280]","[2560, 1280, 1280]",141,True,0.826341
6,gat,6,0.774220,0.914085,0.673874,0.894904,0.865902,0.699684,0.768568,0.897863,...,0.861107,0.686749,64,0.000658,"[2560, 640]","[2560, 1280, 640]","[2560, 1280]",187,False,0.693985
3,gat,3,0.770019,0.911326,0.666797,0.889916,0.863345,0.692831,0.764103,0.887998,...,0.858060,0.679495,16,0.001255,"[2560, 1280, 1280]","[2560, 2560, 2560, 1280, 640]","[2560, 2560, 2560, 1280, 640]",128,True,0.792299
4,gat,4,0.772813,0.888375,0.687571,0.898550,0.862083,0.689836,0.766441,0.878356,...,0.857497,0.679240,16,0.000505,"[2560, 1280, 1280, 640]","[2560, 1280, 1280, 1280]",[2560],106,False,0.236462
8,gat,8,0.782733,0.849595,0.730015,0.901537,0.861813,0.689042,0.779795,0.849559,...,0.859423,0.684378,64,0.000250,[640],"[2560, 1280]",[2560],147,False,0.040616
1,gat,1,0.748706,0.932025,0.631732,0.890051,0.856348,0.681549,0.747996,0.916926,...,0.853912,0.674470,64,0.001077,"[1280, 640]","[2560, 1280]","[2560, 1280, 1280]",149,False,0.321981
2,gat,2,0.715853,0.987607,0.562733,0.874610,0.848205,0.669085,0.710868,0.954176,...,0.842378,0.652868,16,0.001070,"[2560, 2560, 2560, 1280]","[2560, 640]","[2560, 1280]",155,False,0.887857
5,gat,5,0.754755,0.862719,0.679675,0.890614,0.849699,0.662155,0.754248,0.877535,...,0.852220,0.671814,32,0.000344,"[2560, 640]","[2560, 1280]",[2560],193,False,0.587494


In [3]:
import pandas as pd

pd.read_csv("binding_3d_gat_np_classifier.csv").sort_values(by="mcc validation set").iloc[:, :14]

,Model type,Trial,F1_macro_validation_set,Precision validation set,recall validation set,roc_auc validation set,accuracy validation set,mcc validation set,F1_macro_test_set,Precision test set,recall test set,roc_auc test set,accuracy test set,mcc test set
7,gat,7,0.422921,0.296475,0.750000,0.499815,0.473250,-0.005196,0.388387,0.513011,0.750619,0.500309,0.404891,0.006173
1,gat,1,0.308240,0.408819,0.600265,0.500133,0.458399,0.006013,0.334183,0.433373,0.600081,0.500041,0.507490,0.001255
3,gat,3,0.215196,0.745156,0.401900,0.501056,0.531675,0.030041,0.219638,0.662540,0.400951,0.500433,0.541951,0.016766
2,gat,2,0.704474,0.807381,0.728914,0.835050,0.741826,0.549853,0.759712,0.837609,0.756238,0.820431,0.791303,0.581852
6,gat,6,0.788858,0.904968,0.711003,0.908798,0.863566,0.704994,0.729222,0.914461,0.618726,0.877256,0.834749,0.652118
4,gat,4,0.811803,0.845674,0.785852,0.915817,0.870571,0.715181,0.788881,0.851786,0.744205,0.906024,0.857960,0.690879
0,gat,0,0.819681,0.879252,0.772319,0.933679,0.878974,0.730977,0.756237,0.848031,0.691331,0.889565,0.836994,0.646280
5,gat,5,0.825633,0.890540,0.774184,0.928301,0.884326,0.742127,0.805331,0.864072,0.761408,0.920159,0.869888,0.711808


In [6]:
protein_descriptors = read_pickle("propythia_descriptors/features.pkl")["proteins"]
compounds_features = read_pickle("compounds_features.pkl")

In [11]:
import numpy as np

array = np.vstack(protein_descriptors.values())
nan_mask = np.isnan(array)

# Print the boolean mask
print("NaN mask:")
print(nan_mask)

# If you want to find the indices of NaN values
nan_indices = np.argwhere(nan_mask)
print("\nIndices of NaN values:")
print(nan_indices)

NaN mask:
[[False False False ... False False False]
 [False False False ... False False False]
 [False False False ... False False False]
 ...
 [False False False ... False False False]
 [False False False ... False False False]
 [False False False ... False False False]]

Indices of NaN values:
[]


/tmp/ipykernel_2899869/887153175.py:3: FutureWarning: arrays to stack must be passed as a "sequence" type such as list or tuple. Support for non-sequence iterables such as generators is deprecated as of NumPy 1.16 and will raise an error in the future.
  array = np.vstack(protein_descriptors.values())


In [10]:
import numpy as np

array = np.vstack(compounds_features.values())
nan_mask = np.isnan(array)

# Print the boolean mask
print("NaN mask:")
print(nan_mask)

# If you want to find the indices of NaN values
nan_indices = np.argwhere(nan_mask)
print("\nIndices of NaN values:")
print(nan_indices)

NaN mask:
[[False False False ... False False False]
 [False False False ... False False False]
 [False False False ... False False False]
 ...
 [False False False ... False False False]
 [False False False ... False False False]
 [False False False ... False False False]]

Indices of NaN values:
[]


/tmp/ipykernel_2899869/2257703616.py:3: FutureWarning: arrays to stack must be passed as a "sequence" type such as list or tuple. Support for non-sequence iterables such as generators is deprecated as of NumPy 1.16 and will raise an error in the future.
  array = np.vstack(compounds_features.values())


In [1]:
import pandas as pd

dataset = pd.read_csv("augmented_dataset_descriptors_available.csv")
dataset 

/tmp/ipykernel_2933046/672680221.py:3: DtypeWarning: Columns (5,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset = pd.read_csv("augmented_dataset_descriptors_available.csv")


,Sequence,SMILES,Binding,Enzyme ID,Substrate ID,Publication,pubchem_id,Validated,RHEA_ID,EC number,reaction_SMILES
0,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,C(C(=O)O)N,0.0,P00509,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0,True,NaN,NaN,NaN
1,MDYVTLASHAVRQYAPDQIFTASQRAKADAAALGEDAVINATLGEC...,C(C(=O)O)N,0.0,A0A174XK40,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0,True,NaN,NaN,NaN
2,MFQKVDAYAGDPILTLMERFKEDPRSDKVNLSIGLYYNEDGIIPQL...,C(C(=O)O)N,0.0,P04693,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0,True,NaN,NaN,NaN
3,MFQKVDAYAGDPILSLMERFKEDPRSDKVNLSIGLYYNDDGIIPQL...,C(C(=O)O)N,0.0,A0A3N0RMA5,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0,True,NaN,NaN,NaN
4,MSPIEKSSKLENVCYDIRGPVLKEAKRLEEEGNKVLKLNIGNPAPF...,C(C(=O)O)N,0.0,P0A959,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,750.0,True,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
819916,MSEAEARPTNFIRQIIDEDLASGKHTTVHTRFPPEPNGYLHIGHAK...,OC[C@@H](O)[C@H](O)[C@@H](O)CO,0.0,P00962,CHEBI:17151,NaN,NaN,False,NaN,NaN,NaN
819917,MEPEAPRRRHTHQRGYLLTRNPHLNKDLAFTLEERQQLNIHGLLPP...,Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)([O-])OP(=O...,0.0,P48163,CHEBI:30616,NaN,NaN,False,NaN,NaN,NaN
819918,DGLRGGGRFFHGLAWILEEPWSSWFILNLKTTAEEEKEEVESLKSI...,COC1=C(OC)C(=O)C(CC=C(C)C)=C(C)C1=O,0.0,Q3TIG7,CHEBI:16389,NaN,NaN,False,NaN,NaN,NaN
819919,DGLRGGGRFFHGLAWILEEPWSSWFILNLKTTAEEEKEEVESLKSI...,N[C@@H](CCC([O-])=N[C@@H](CS)C(O)=NCC(=O)O)C(=O)O,0.0,Q3TIG7,CHEBI:57925,NaN,NaN,False,NaN,NaN,NaN


In [3]:
dataset[dataset["Validated"]==True].to_csv("augmented_dataset_descriptors_available_validated.csv", index=False)